# AIBackends - local PII redaction with GLiNER

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-GLiNER-PII-redaction.ipynb)

Detect and redact personal data locally with the
[nvidia/gliner-pii](https://huggingface.co/nvidia/gliner-pii) backend in
`aibackends[pii]`. Covers `examples/tasks/redact_text.py` and `redact_text_batch.py`:
the one-call helper, a reusable `RedactPIITask` with custom labels, and batch
redaction that loads the model once.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
%pip install -q "aibackends[pii]>=0.8.1"

# If a later import fails with a transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [2]:
import aibackends

# aibackends accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch

    DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print("aibackends", aibackends.__version__)
print("device:", DEVICE)

aibackends 0.8.1
device: cpu


In [3]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/donvito/aibackends/main/examples/data"
DATA_DIR = Path("aibackends_data")


def fetch(relative_path: str) -> Path:
    """Download a sample file from the aibackends examples once and return its path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        try:
            urlretrieve(f"{DATA_URL}/{relative_path}", partial)
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    return path

## 1. Load the GLiNER PII model (on GPU when available)

The backend caches one model per process. We warm it once and move it to CUDA on a GPU
runtime, so every later redaction call reuses the GPU copy.

In [4]:
import time

from aibackends.backends.pii import get_pii_backend, list_pii_backends
from aibackends.backends.pii.gliner import PII_BACKEND_SPEC, load_gliner_model

print("PII backends:", list_pii_backends())
t = time.perf_counter()
gliner_model = load_gliner_model(PII_BACKEND_SPEC)
if DEVICE == "gpu":
    gliner_model.to("cuda")
print(f"loaded {PII_BACKEND_SPEC.model_id} in {time.perf_counter() - t:.1f}s")

PII backends: ['gliner', 'openai-privacy']


loaded nvidia/gliner-pii in 25.2s


## 2. One-call redaction

`redact_pii` returns the redacted text, every entity found, and a placeholder-to-original map.

In [5]:
from aibackends.tasks import redact_pii

message = "Hi, I'm Jane Doe (jane.doe@example.com, +1 415 555 0199). Please update my address."
result = redact_pii(message, labels=["person_name", "email", "phone_number"])
print(result.redacted_text)
for entity in result.entities_found:
    print(f"  {entity.entity_type:<14} {entity.text!r}")
print(result.redaction_map)

Hi, I'm [PERSON_NAME_1] [PERSON_NAME_2] ([EMAIL_3], [PHONE_NUMBER_4]). Please update my address.
  PERSON_NAME    'Jane'
  PERSON_NAME    'Doe'
  EMAIL          'jane.doe@example.com'
  PHONE_NUMBER   '+1 415 555 0199'
{'Jane': '[PERSON_NAME_1]', 'Doe': '[PERSON_NAME_2]', 'jane.doe@example.com': '[EMAIL_3]', '+1 415 555 0199': '[PHONE_NUMBER_4]'}


## 3. A reusable task with custom labels (`redact_text.py`)

GLiNER is zero-shot, so the label list is yours to choose. Tasks accept a file path or a string.

In [6]:
from aibackends.tasks import RedactPIITask, create_task

redact_contract = create_task(
    RedactPIITask,
    backend="gliner",
    labels=[
        "name", "email", "phone_number", "address",
        "identification_number", "passport_number", "account_number",
    ],
)
contract = redact_contract.run(fetch("contract.txt"))
print(contract.redacted_text[:1200])
print(f"\n{len(contract.entities_found)} entities redacted")

RESIDENTIAL RENTAL AGREEMENT

This Rental Agreement (“Agreement”) is made and entered into on 1 May 2026, between:

Landlord: [NAME_1] [NAME_2]
ID No.: [IDENTIFICATION_NUMBER_3]
Address: [ADDRESS_4]
Email: [EMAIL_5]
Phone: [PHONE_NUMBER_6]

and

Tenant: [NAME_7] [NAME_8]
Passport No.: [PASSPORT_NUMBER_9]
Address: [ADDRESS_10]
Email: [EMAIL_11]
Phone: [PHONE_NUMBER_12]

⸻

1. Property

The Landlord hereby rents to the Tenant the property located at:
[ADDRESS_13] (“Premises”).

⸻

2. Term

The lease shall begin on 1 June 2026 and end on 31 May 2027, unless terminated earlier in accordance with this Agreement.

⸻

3. Rent

* Monthly Rent: SGD 3,200
* Due Date: 1st of each month
* Payment Method: Bank Transfer to Account [ACCOUNT_NUMBER_14]

Late payments may incur a fee of SGD 50 after 5 days.

⸻

4. Security Deposit

The Tenant agrees to pay a security deposit of SGD 3,200 before moving in.
This deposit will be returned within 14 days after lease termination, subject to deductions for da

## 4. Batch redaction with one model load (`redact_text_batch.py`)

In [7]:
backend = get_pii_backend("gliner")
backend.load()

PII_LABELS = ["person_name", "email", "phone_number", "address"]
for relative in ["batch/pii_note_1.txt", "batch/pii_note_2.txt", "batch/pii_note_3.txt"]:
    text = fetch(relative).read_text(encoding="utf-8")
    t = time.perf_counter()
    redacted = backend.redact(text, labels=PII_LABELS)
    ms = (time.perf_counter() - t) * 1000
    print(f"--- {relative}: {len(redacted.entities_found)} entities in {ms:.0f} ms")
    print(redacted.redacted_text, "\n")

--- batch/pii_note_1.txt: 5 entities in 1382 ms
Customer record for [PERSON_NAME_1] [PERSON_NAME_2].
Reach her at [EMAIL_3] or [PHONE_NUMBER_4].
Mail the revised agreement to [ADDRESS_5].
 



--- batch/pii_note_2.txt: 4 entities in 577 ms
[PERSON_NAME_1] asked for a follow-up before the renewal call.
Send the quote to [EMAIL_2] and call him on [PHONE_NUMBER_3].
The billing contact address is [ADDRESS_4].
 



--- batch/pii_note_3.txt: 4 entities in 230 ms
[PERSON_NAME_1] shared onboarding details for the vendor review.
Her email is [EMAIL_2] and her mobile is [PHONE_NUMBER_3].
Ship the welcome pack to [ADDRESS_4].
 



## 5. From the CLI

In [8]:
!aibackends task redact-pii --input "Email john.smith@acme.com or call 555-0100." --backend gliner

  warnings.warn(


  warnings.warn(
Fetching 13 files: 100%|█████████████████████| 13/13 [00:00<00:00, 1830.47it/s]


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{
  "original_text": "Email john.smith@acme.com or call 555-0100.",
  "redacted_text": "Email [EMAIL_1] or call [PHONE_NUMBER_2].",
  "entities_found": [
    {
      "entity_type": "EMAIL",
      "text": "john.smith@acme.com",
      "start": 6,
      "end": 25,
      "replacement": "[EMAIL_1]"
    },
    {
      "entity_type": "PHONE_NUMBER",
      "text": "555-0100",
      "start": 34,
      "end": 42,
      "replacement": "[PHONE_NUMBER_2]"
    }
  ],
  "redaction_map": {
    "john.smith@acme.com": "[EMAIL_1]",
    "555-0100": "[PHONE_NUMBER_2]"
  },
  "backend_used": "gliner"
}
